# PRUEBAS DE TABLAS FINALES GOLD

### 1. Ventas netas por sucursal y mes.

### 2. Productos, sobre todo el período:

### 3. Dotación

### 4. Ventas por vendedor

In [0]:
%sql
WITH ventas_base AS(
    SELECT venta_id, vendedor_id, cantidad, monto_total, fecha_venta FROM electrocasa_dev.silver.ventas
    WHERE vendedor_id IS NOT NULL AND canal = 'TIENDA_FISICA'
),
empleados as(
    SELECT ep.id_empleado, ep.nombre, ep.sucursal_id, s.nombre_sucursal, ep.__START_AT, ep.__END_AT FROM electrocasa_dev.silver.empleados ep
    INNER JOIN electrocasa_dev.silver.sucursales s ON s.sucursal_id = ep.sucursal_id
),
vista_cruzada AS(
    SELECT EXTRACT(YEAR FROM vb.fecha_venta) anho, EXTRACT(MONTH FROM vb.fecha_venta) mes,vb.venta_id, vb.vendedor_id,
    CASE
        WHEN ep.nombre IS NULL THEN 'SIN ASIGNAR'
        ELSE ep.nombre
    END nombre
    , vb.cantidad, vb.monto_total, ep.sucursal_id, 
    CASE
        WHEN ep.nombre_sucursal IS NULL THEN 'SIN ASIGNAR'
        ELSE ep.nombre_sucursal
    END sucursal
    FROM ventas_base vb
    LEFT JOIN empleados ep ON vb.vendedor_id = ep.id_empleado AND vb.fecha_venta >= ep.__START_AT AND vb.fecha_venta < ep.__END_AT
),
vista_resultado AS(
    SELECT 
    anho, 
    mes,
    vendedor_id,
    sucursal,
    COUNT(venta_id) nro_ventas,
    SUM(cantidad) cantidad
    FROM vista_cruzada
    GROUP BY 1, 2, 3, 4
    ORDER BY 1, 2, 3, 4
)
SELECT * FROM vista_resultado 

In [0]:
%sql
SELECT * FROM electrocasa_dev.bronze.ventas 

In [0]:
%sql
SELECT * FROM electrocasa_dev.silver.empleados

In [0]:
%sql
SELECT * FROM electrocasa_dev.silver.sucursales

### 5. Reseñas

In [0]:
%sql
SELECT rs.* FROM electrocasa_dev.silver.resenas rs
LEFT JOIN electrocasa_dev.silver.catalogo ct ON rs.producto_id = ct.producto_id
WHERE ct.producto_id IS NULL

In [0]:
%sql
WITH vista_total AS(
    SELECT
    CONCAT(
        EXTRACT(YEAR FROM rs.fecha_resena),
        '-',
        EXTRACT(MONTH FROM rs.fecha_resena),
        '-',
        ct.categoria
    ) ID,
    ct.categoria,
    EXTRACT(YEAR FROM rs.fecha_resena) anho,
    EXTRACT(MONTH FROM rs.fecha_resena) mes,
    COUNT(rs.resena_id) num_resenas_total
    FROM electrocasa_dev.silver.resenas rs
    INNER JOIN electrocasa_dev.silver.catalogo ct ON rs.producto_id = ct.producto_id
    GROUP BY 1,2,3,4
    ORDER BY 1,2,3,4
),
vista_2 AS(
    SELECT
    CONCAT(
        EXTRACT(YEAR FROM rs.fecha_resena),
        '-',
        EXTRACT(MONTH FROM rs.fecha_resena),
        '-',
        ct.categoria
    ) ID,
    EXTRACT(YEAR FROM rs.fecha_resena) anho,
    EXTRACT(MONTH FROM rs.fecha_resena) mes,
    ct.categoria,
    CASE
        WHEN rs.calificacion IN (1, 2) THEN 'NEGATIVO'
        ELSE 'POSITIVO'
    END categoria_calificacion,
    COUNT(rs.resena_id) num_resenas
    FROM electrocasa_dev.silver.resenas rs
    INNER JOIN electrocasa_dev.silver.catalogo ct ON rs.producto_id = ct.producto_id
    GROUP BY 1,2,3,4,5
    ORDER BY 1,2,3,4,5
),
vista_cruzada AS(
    SELECT
        v2.anho,
        v2.mes,
        v2.categoria,
        v2.categoria_calificacion,
        v2.num_resenas,
        vt.num_resenas_total,
        v2.num_resenas / vt.num_resenas_total * 100 porcentaje
    FROM vista_2 v2
    LEFT JOIN vista_total vt ON v2.ID = vt.ID
    ORDER BY 1,2,3,4
),
vista_resultado AS(
    SELECT anho, mes, categoria, categoria_calificacion, porcentaje FROM vista_cruzada WHERE categoria_calificacion = 'NEGATIVO'
)
SELECT * FROM vista_resultado

In [0]:
%sql
SELECT * FROM electrocasa_dev.silver.catalogo

### 6. Despachos